# NB_lab_test_trends

Builds `adb_caresyncs.gold.lab_test_trends` (one row per test type per month) from silver lab_results per the STTM *Silver to Gold* v1.0, with calendar month/quarter names and a month-over-month abnormal-rate trend via LAG(). Full rebuild via CREATE OR REPLACE TABLE AS SELECT.

In [0]:
%sql
-- Target : adb_caresyncs.gold.lab_test_trends
-- Grain  : one row per test_name per calendar month (driving table silver.lab_results)
-- Load   : FULL rebuild; LEFT JOIN to calendar on the month's first day for month/quarter names
-- Trend  : LAG(abnormal_rate_pct) over test_name ordered by year, month
CREATE OR REPLACE TABLE adb_caresyncs.gold.lab_test_trends
COMMENT 'Gold: one row per lab test type per month with volumes, abnormal/critical rates and month-over-month trend (STTM silver to gold v1.0)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                            AS year,
    month(test_date)                           AS month,
    CAST(date_trunc('MONTH', test_date) AS DATE) AS month_start_date,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM adb_caresyncs.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date), CAST(date_trunc('MONTH', test_date) AS DATE)
),
rates AS (
  SELECT
    m.*,
    CAST(m.abnormal_test_count * 100.0 / NULLIF(m.total_tests, 0) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(m.critical_test_count * 100.0 / NULLIF(m.total_tests, 0) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_trend AS (
  SELECT
    r.*,
    LAG(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  t.test_name,
  CAST(t.year AS INT)                                                    AS year,
  CAST(t.month AS INT)                                                   AS month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)                                             AS total_tests,
  CAST(t.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                     AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                   AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_trend t
LEFT JOIN adb_caresyncs.silver.calendar c
  ON c.date_key = CAST(date_format(t.month_start_date, 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (e.g. ADF / job task value)
updated_df = spark.read.table("adb_caresyncs.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))